In [4]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://en.wikipedia.org/wiki/Artificial_intelligence"

loader = WebBaseLoader(url)

documents = loader.load()

C:\Users\WINDOWS\AppData\Local\Temp\ipykernel_3084\1240642063.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks = text_splitter.split_documents(documents)

In [48]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

key = os.getenv("GOOGLE_API_KEY")

print("Key loaded:", bool(key))
print("Key length:", len(key) if key else 0)

Key loaded: True
Key length: 53


In [119]:
import os
from dotenv import load_dotenv

load_dotenv()

print("Groq key exists:", os.getenv("GROQ_API_KEY") is not None)

Groq key exists: True


In [103]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=key
)

In [120]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [121]:
response = llm.invoke(
    "Say 'LLM is working' in exactly three words."
)

print(response.content)

LLM is working


In [8]:
vector = embeddings.embed_query(
    chunks[0].page_content
)

print(type(vector))
print(len(vector))

<class 'list'>
3072


In [9]:
# print("Total chunks:", len(chunks))
test_chunks = chunks[:20]

print("Using chunks:", len(test_chunks))

Using chunks: 20


In [11]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=test_chunks,
    embedding=embeddings
)

print("FAISS vector store created successfully")

FAISS vector store created successfully


In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [13]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful knowledge assistant.

Answer the user's question using only the provided context.

If the answer cannot be found in the context,
say that the information was not found in the
provided knowledge base.

Context:
{context}

Question:
{input}

Answer:
""")

In [14]:
def rag_answer(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    response = llm.invoke(
        prompt.format(
            context=context,
            input=question
        )
    )

    sources = []

    for doc in docs:
        source = doc.metadata.get("source")

        if source and source not in sources:
            sources.append(source)

    return {
        "answer": response.content,
        "context": docs,
        "sources": sources
    }

In [15]:
def chat_with_rag(question):
    standalone_question = rewrite_question(question)

    docs = retriever.invoke(standalone_question)

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    history_text = "\n".join(
        f"User: {user}\nAssistant: {assistant}"
        for user, assistant in chat_history
    )

    chat_prompt = f"""
You are a helpful knowledge assistant.

Answer the user's question using only the provided context.

Use the conversation history to understand references
to previous messages.

If the answer cannot be found in the provided context,
say that the information was not found in the provided
knowledge base.

Conversation history:
{history_text}

Context:
{context}

Current question:
{question}

Answer:
"""

    response = llm.invoke(chat_prompt)

    answer = response.content

    chat_history.append((question, answer))

    sources = []

    for doc in docs:
        source = doc.metadata.get("source")

        if source and source not in sources:
            sources.append(source)

    return {
        "answer": answer,
        "sources": sources,
        "context": docs,
        "standalone_question": standalone_question
    }

In [16]:
from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_template("""
Given the conversation history and the user's latest question,
rewrite the latest question as a standalone question.

Do not answer the question.
Only return the rewritten question.

Conversation history:
{history}

Latest question:
{question}

Standalone question:
""")

In [17]:
def rewrite_question(question):
    history_text = "\n".join(
        f"User: {user}\nAssistant: {assistant}"
        for user, assistant in chat_history
    )

    response = llm.invoke(
        rewrite_prompt.format(
            history=history_text,
            question=question
        )
    )

    return response.content.strip()

In [18]:
chat_history = []

In [31]:
response = chat_with_rag(
    "What is artificial intelligence?"
)

print(response["answer"])

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.


In [30]:
response = chat_with_rag(
    "What are its applications?"
)

print(response["answer"])

Applications of AI include:
*   Advanced web search engines
*   Chatbots
*   Virtual assistants
*   Autonomous vehicles
*   Play and analysis in strategy games (e.g., chess and Go)
*   Content generation (e.g., images, audio, and videos)
*   AI-assisted software development
*   Physical AI
*   Health and medicine
*   Gaming
*   Mathematics
*   Finance
*   Military
*   Generative AI
*   Agents
*   Web search
*   Sexuality
*   Other industry-specific tasks
*   Art
*   Music
*   Bioinformatics
*   Deepfake
*   Earth sciences
*   Government
*   Healthcare
*   Industry
*   Software development
*   Translation
*   Physics


In [94]:
print(chat_history)

[('What is artificial intelligence?', 'Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.'), ('What are its applications?', 'High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., images, audio, and videos). Other applications include AI-assisted software development, physical AI, health and medicine, gaming, mathematics, finance, military, generative AI, agents, and sexuality, as well as other indust

In [ ]:
standalone_question = rewrite_question(
    "Which of these are used in healthcare?"
)

print(standalone_question)

Which applications of artificial intelligence are used in healthcare?


In [29]:
response = chat_with_rag(
    "What are its applications?"
)

print(response["standalone_question"])
print()
print(response["answer"])

What are the applications of the subject?

Applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g. images, audio, and videos).

More specifically, the applications listed are:
*   AI-assisted software development
*   Chatbots
*   Physical AI
*   Health and medicine
*   Gaming
*   Mathematics
*   Finance
*   Military
*   Generative AI
*   Agents
*   Web search
*   Sexuality
*   Other industry-specific tasks

Formal knowledge representations are used in content-based indexing and retrieval, scene interpretation, clinical decision support, and knowledge discovery.


In [22]:
def load_and_split_url(url):
    loader = WebBaseLoader(url)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )

    chunks = text_splitter.split_documents(documents)

    return chunks

In [24]:
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"

new_chunks = load_and_split_url(url)

print("Documents loaded:", len(new_chunks))

Documents loaded: 301


In [25]:
def create_vectorstore(chunks):
    vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )
    
    return vectorstore

In [26]:
practice_chunks = new_chunks[:20]

vectorstore = create_vectorstore(practice_chunks)

print("Vector store created.")

Vector store created.


In [27]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [28]:
results = retriever.invoke(
    "What is artificial intelligence?"
)

print("Retrieved:", len(results))

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:500])

Retrieved: 3

--- Result 1 ---
History
Timeline
Progress
AI winter
AI boom
AI bubble

Controversies
Deepfake pornography
Taylor Swift deepfake pornography controversy
Grok sexual deepfake scandal
Google Gemini image generation controversy
It's the Most Terrible Time of the Year
Pause Giant AI Experiments
Removal of Sam Altman from OpenAI
Statement on AI Risk
Tay (chatbot)
Théâtre D'opéra Spatial
Voiceverse NFT plagiarism scandal

Glossaryvte


Artificial intelligence (AI) is the capability of computational systems to perform 

--- Result 2 ---
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g. images, audio, and videos).

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI resear

Flow

In [2]:
url = input("Enter website URL: ")

In [35]:
new_chunks = load_and_split_url("https://en.wikipedia.org/wiki/Machine_learning")

print("Total chunks:", len(new_chunks))

Total chunks: 171


In [33]:
practice_chunks = new_chunks[:20]

In [ ]:
vectorstore = create_vectorstore(practice_chunks)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}
)

In [36]:
response = rag_answer(
    "What is machine learning?"
)

print(response["answer"])

Machine learning is the study of programs that can improve their performance on a given task automatically.


In [37]:
chat_history = []

response = chat_with_rag(
    "What is machine learning?"
)

print(response["answer"])

Machine learning is the study of programs that can improve their performance on a given task automatically.


In [42]:
chat_history = []

response = chat_with_rag(
    "Who coined the term artificial intelligence?"
)

print(response["answer"])

The information was not found in the provided knowledge base.


# Reusable code: for web url

In [26]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\working of mineee 2026\JP morgan\rag


In [4]:
from app.loaders.web_loader import load_web_url
from app.ingestion.splitter import split_documents

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"

documents = load_web_url(url)

print("Documents:", len(documents))

Documents: 1


In [6]:
chunks = split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 301


# pdf

In [7]:
from app.loaders.pdf_loader import load_pdf

In [8]:
pdf_path = "../data/uploads/test.pdf"

pdf_documents = load_pdf(pdf_path)

print("PDF pages:", len(pdf_documents))

PDF pages: 23


In [10]:
print(type(pdf_documents[0]))
print(pdf_documents[0].metadata)

<class 'langchain_core.documents.base.Document'>
{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-02-23T09:47:07+05:30', 'author': 'Loknath Y', 'moddate': '2026-02-23T09:47:07+05:30', 'source': '../data/uploads/test.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}


In [11]:
pdf_chunks = split_documents(pdf_documents)

print("PDF chunks:", len(pdf_chunks))

PDF chunks: 23


# document

In [16]:
from app.loaders.docx_loader import load_docx

In [18]:
docx_path = "../data/uploads/test.docx"

docx_documents = load_docx(docx_path)

print("Documents:", len(docx_documents))
print(type(docx_documents[0]))
print(docx_documents[0].page_content[:1000])
print(docx_documents[0].metadata)

Documents: 1
<class 'langchain_core.documents.base.Document'>
DATA SCIENCE

INTERNSHIP ROADMAP

2-Month Structured Plan to Land a High-Package DS Internship

Designed for 3rd Year Engineering Students



Python  |  Statistics  |  SQL  |  DSA  |  Machine Learning  |  Projects




1. Roadmap Overview

This is a focused, no-fluff 2-month plan to get you internship-ready as a Data Scientist. Every day is structured around 6 hours of deliberate practice across core topics, DSA, and SQL — the three pillars of every DS internship interview.



Phase

Weeks

Focus

Month 1

Weeks 1–4

Python, Statistics, Math for ML, Data Visualization, EDA, SQL basics, DSA foundations

Month 2

Weeks 5–8

ML algorithms, Model evaluation, End-to-end projects, Deployment basics, Interview prep, Applications



Internship-Ready Checklist (by Day 60)

GitHub with 2–3 polished end-to-end data science projects

DS-focused ATS-ready resume

80+ SQL problems solved (HackerRank / LeetCode SQL)

50+ LeetCode DSA proble

In [19]:
docx_chunks = split_documents(docx_documents)

print("DOCX chunks:", len(docx_chunks))

DOCX chunks: 15


# text

In [29]:
from pathlib import Path

path = Path("../app/loaders/text_loader.py")

print(path.resolve())
print(path.read_text(encoding="utf-8"))

D:\working of mineee 2026\JP morgan\rag\app\loaders\text_loader.py
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document


def load_text(file_path: str):
    loader = TextLoader(
        file_path,
        encoding="utf-8"
    )
    return loader.load()


def load_pasted_text(text: str):
    return [
        Document(
            page_content=text,
            metadata={"source": "pasted_text"}
        )
    ]


In [30]:
import importlib
import app.loaders.text_loader as text_loader

importlib.reload(text_loader)

print(dir(text_loader))

['Document', 'TextLoader', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'load_pasted_text', 'load_text']


In [31]:
from app.loaders.text_loader import load_text, load_pasted_text

In [32]:
txt_documents = load_text("../data/uploads/test.txt")

print("Documents:", len(txt_documents))
print(type(txt_documents[0]))
print(txt_documents[0].page_content)
print(txt_documents[0].metadata)

Documents: 1
<class 'langchain_core.documents.base.Document'>
JPMorgan Chase Code for Good Hackathon

 

ABOUT THE EVENT: Teams of students will compete against each other on behalf of a charity assigned to them in order to provide a technological solution to a problem that the charity faces. The team or teams which, in the opinion of JPMorgan Chase Bank, N.A. (“JPMC”), make the best attempt at developing such a solution will receive a prize for themselves and their assigned charity. All prizes are awarded “as is” and JPMC, to the fullest extent permitted by law, makes no warranty or condition in respect of any prize or part thereof. JPMC reserves the right to change any prize at its discretion.

 

All entries become the exclusive property of JPMC and will not be acknowledged or returned. By entering the Code for Good Hackathon (“Hackathon”), entrants accept and agree to be bound by these Hackathon Official Rules (the “Rules”) and the decisions of the judges, which shall be final, bin

In [22]:
txt_chunks = split_documents(txt_documents)

print("TXT chunks:", len(txt_chunks))

TXT chunks: 20


In [33]:
pasted_documents = load_pasted_text(
    """
    Our organization provides educational programs
    for students and volunteers.
    """
)

print(pasted_documents[0].page_content)
print(pasted_documents[0].metadata)


    Our organization provides educational programs
    for students and volunteers.
    
{'source': 'pasted_text'}


In [34]:
pasted_chunks = split_documents(pasted_documents)

print("Pasted text chunks:", len(pasted_chunks))
print(pasted_chunks[0].page_content)

Pasted text chunks: 1
Our organization provides educational programs
    for students and volunteers.


# ingestion test

In [35]:
from app.ingestion.ingestion_service import load_source

In [36]:
documents = load_source(
    "https://en.wikipedia.org/wiki/Artificial_intelligence"
)

print("Documents:", len(documents))
print(type(documents[0]))

Documents: 1
<class 'langchain_core.documents.base.Document'>


In [37]:
pdf_documents = load_source(
    "../data/uploads/test.pdf"
)

print("PDF documents:", len(pdf_documents))

PDF documents: 23


In [38]:
docx_documents = load_source(
    "../data/uploads/test.docx"
)

print("DOCX documents:", len(docx_documents))

DOCX documents: 1


In [39]:
txt_documents = load_source(
    "../data/uploads/test.txt"
)

print("TXT documents:", len(txt_documents))

TXT documents: 1


In [42]:
from app.ingestion.ingestion_service import load_source, load_text_input

In [43]:
pasted_documents = load_text_input(
    """
    Our organization provides educational programs
    for students and volunteers.
    """
)

print(pasted_documents[0].page_content)


    Our organization provides educational programs
    for students and volunteers.
    


In [44]:
chunks = split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 301


# vector embeddings

In [140]:
from app.vectorstore.knowledge_base import create_knowledge_base

In [51]:
print(type(embeddings))

<class 'langchain_google_genai.embeddings.GoogleGenerativeAIEmbeddings'>


In [52]:
pasted_documents = load_text_input(
    """
    Our organization provides educational programs
    for students and volunteers.
    """
)

pasted_chunks = split_documents(pasted_documents)

print("Chunks:", len(pasted_chunks))

Chunks: 1


In [53]:
vectorstore = create_knowledge_base(
    pasted_chunks,
    embeddings
)

print("Knowledge base created successfully")

Knowledge base created successfully


# for retrieving

In [54]:
from app.retrieval.retriever import create_retriever

In [55]:
retriever = create_retriever(
    vectorstore,
    k=3
)

In [58]:
results = retriever.invoke(
    "What does the organization provide?"
)

print("Retrieved:", len(results))

for doc in results:
    print(doc.page_content)
    print(doc.metadata)

Retrieved: 1
Our organization provides educational programs
    for students and volunteers.
{'source': 'pasted_text'}


In [57]:
test_vector = embeddings.embed_query(
    "This is a small test sentence."
)

print("Embedding successful")
print("Vector dimensions:", len(test_vector))

Embedding successful
Vector dimensions: 3072


# retrival test for faiss

In [59]:
pasted_documents = load_text_input(
    """
    Our organization provides educational programs
    for students and volunteers.
    """
)

pasted_chunks = split_documents(pasted_documents)

vectorstore = create_knowledge_base(
    pasted_chunks,
    embeddings
)

print("Knowledge base created")

Knowledge base created


In [79]:
import importlib
import app.vectorstore.knowledge_base as kb

importlib.reload(kb)

<module 'app.vectorstore.knowledge_base' from 'd:\\working of mineee 2026\\JP morgan\\rag\\app\\vectorstore\\knowledge_base.py'>

In [93]:
print(hasattr(kb, "create_knowledge_base_from_sources"))

True


In [80]:
from app.vectorstore.knowledge_base import (
    create_knowledge_base,
    save_knowledge_base,
    load_knowledge_base
)

In [81]:
save_knowledge_base(vectorstore)

Knowledge base saved to: data/faiss


# Load FAISS from disk

In [82]:
from app.vectorstore.knowledge_base import load_knowledge_base

loaded_vectorstore = load_knowledge_base(embeddings)

print("Knowledge base loaded successfully")

Knowledge base loaded successfully


In [83]:
retriever = create_retriever(
    loaded_vectorstore,
    k=3
)

In [84]:
results = retriever.invoke(
    "What does the organization provide?"
)

print("Retrieved documents:", len(results))

for i, doc in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(doc.page_content)
    print("Source:", doc.metadata)

Retrieved documents: 1

--- Result 1 ---
Our organization provides educational programs
    for students and volunteers.
Source: {'source': 'pasted_text'}


# FAISS database

In [85]:
import importlib
import app.ingestion.ingestion_service as ingestion_service

importlib.reload(ingestion_service)

<module 'app.ingestion.ingestion_service' from 'd:\\working of mineee 2026\\JP morgan\\rag\\app\\ingestion\\ingestion_service.py'>

In [86]:
from app.ingestion.ingestion_service import (
    load_source,
    load_text_input
)

In [87]:
documents = load_source(
    "https://en.wikipedia.org/wiki/Artificial_intelligence"
)

print(documents[0].metadata)

{'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en', 'source_type': 'web'}


In [88]:
pdf_documents = load_source(
    "../data/uploads/test.pdf"
)

print(pdf_documents[0].metadata)

{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-02-23T09:47:07+05:30', 'author': 'Loknath Y', 'moddate': '2026-02-23T09:47:07+05:30', 'source': 'test.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_type': 'pdf'}


In [89]:
text_documents = load_text_input(
    "Our organization provides educational programs."
)

print(text_documents[0].metadata)

{'source': 'pasted_text', 'source_type': 'text'}


In [90]:
chunks = split_documents(documents)

print(chunks[0].metadata)

{'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en', 'source_type': 'web'}


# Build the combined knowledge base

In [94]:
from app.vectorstore.knowledge_base import create_knowledge_base_from_sources
from app.ingestion.splitter import split_documents

In [96]:
sources = [
    "../data/uploads/test.pdf",
    "../data/uploads/test.txt"
]

In [97]:
multi_vectorstore, multi_chunks = create_knowledge_base_from_sources(
    sources=sources,
    embeddings=embeddings,
    splitter=split_documents
)

print("Total chunks:", len(multi_chunks))

Total chunks: 43


In [98]:
for i, chunk in enumerate(multi_chunks):
    print(f"\n--- Chunk {i + 1} ---")
    print("Source:", chunk.metadata.get("source"))
    print("Type:", chunk.metadata.get("source_type"))
    print("Content:", chunk.page_content[:300])


--- Chunk 1 ---
Source: test.pdf
Type: pdf
Content: DEPARTMENT OF AERONAUTICAL ENGINEERING 
TOP 10 STUDENTS (UPTO III SEMESTER) – 2024 ADMITTED BATCH 
 
S.No Roll Number Student Name CGPA 
1 24951A2143 SADHU SATHVIKA 8.97 
2 24951A2147 CHIRRA SHALINI 8.77 
3 24951A2164 MADAMANCHI YUKTHA CHOWDARY 8.73 
4 24951A2156 KUMMARI SWETHA 8.60 
5 25955A2102 BA

--- Chunk 2 ---
Source: test.pdf
Type: pdf
Content: DEPARTMENT OF MECHANICAL ENGINEERING 
TOP 10 STUDENTS (UPTO III SEMESTER) – 2024 ADMITTED BATCH 
 
S.No Roll Number Student Name CGPA 
1 24951A0311 KORRA KIRAN 8.85 
2 25955A0304 KARRA RAVITEJA 8.70 
3 24951A0306 DASARI DEVAKINANDAN 8.48 
4 25955A0305 MADDERLA RITHIKA 8.40 
5 24951A0302 GANJI ABHINA

--- Chunk 3 ---
Source: test.pdf
Type: pdf
Content: DEPARTMENT OF CIVIL ENGINEERING 
TOP 10 STUDENTS (UPTO III SEMESTER) – 2024 ADMITTED BATCH 
 
S.No Roll Number Student Name CGPA 
1 24951A0116 BANOTHU SAGAR 8.78 
2 25955A0108 BANDI SOWMYA 8.70 
3 25955A0101 DUDAPAKA ABHIRAM 8.50 
4 24951A

In [99]:
from app.retrieval.retriever import create_retriever

multi_retriever = create_retriever(
    multi_vectorstore,
    k=3
)

In [100]:
results = multi_retriever.invoke(
    "What educational programs does the organization provide?"
)

for doc in results:
    print("\nCONTENT:")
    print(doc.page_content)

    print("SOURCE:")
    print(doc.metadata)


CONTENT:
SPONSORS:

 

JPMorgan Chase Bank, N.A. 270 Park Avenue, New York, New York 10017
J.P. Morgan Chase Bank – Hong Kong Branch. Chater House, 8 Connaught Road Central, Hong Kong
J.P. Morgan Chase Bank – Singapore Branch. 168 Robinson Road, Singapore
J.P. Morgan Services Argentina S.R.L., Av. Belgrano 955, 1st Floor, C.A.B.A, Argentina
J.P. Morgan Services India Private Limited, Prism Towers, Levels 9 to11, Mindspace, Link Road, Goregaon (West), Mumbai 400 104, India






I, or my parent or legal guardian (if Participant is under eighteen years of age or an incapacitated adult), hereby grant JPMorgan Chase Bank, N.A. (together with its affiliates, employees and agents, “JPMC”), permission to use my image, likeness, name, voice, statements and/or quotes/testimonials in original or altered/edited form, in any and all media (e.g., TV, internet, social media, etc.), in connection with any promotion, advertisement or other conduct of trade.
SOURCE:
{'source': 'test.txt', 'source_type

In [101]:
results = multi_retriever.invoke(
    "How can volunteers register?"
)

for doc in results:
    print("\nCONTENT:")
    print(doc.page_content)

    print("SOURCE:")
    print(doc.metadata)


CONTENT:
I acknowledge that my participation is entirely voluntary and as a result, I will receive no financial compensation. I understand that I do not own the copyright or have any rights of ownership or other claim over any of the content that may include my name, image, likeness, and/or voice that may be in any media, and that no royalties or other compensation of any kind shall be payable to me.

I hereby hold harmless and release JPMC from any liabilities or losses relating to this Appearance Release Form.

I have read this Appearance Release Form, and am fully familiar with its contents and agree to its terms. I further understand that JPMC may rely on this Appearance Release Form as a defense to any claim or action that I may bring against JPMC relating to this Release and Authorization. I agree that either an electronic or physical copy of this Appearance Release Form bearing my actual or digital signature shall be considered an original document.
SOURCE:
{'source': 'test.txt

# Rag function importing 

In [111]:
from app.rag.rag_service import generate_rag_answer

In [128]:
import importlib
import app.rag.rag_service as rag_service

importlib.reload(rag_service)

from app.rag.rag_service import generate_rag_answer

In [113]:
multi_retriever = create_retriever(
    multi_vectorstore,
    k=3
)

In [116]:
results = multi_retriever.invoke(
    "Who can participate in JPMC Code for Good?"
)

print("Retrieved:", len(results))

for i, doc in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print("SOURCE:", doc.metadata.get("source"))
    print("TYPE:", doc.metadata.get("source_type"))
    print("CONTENT:")
    print(doc.page_content)

Retrieved: 3

--- Result 1 ---
SOURCE: test.txt
TYPE: txt
CONTENT:
JPMorgan Chase Code for Good Hackathon

 

ABOUT THE EVENT: Teams of students will compete against each other on behalf of a charity assigned to them in order to provide a technological solution to a problem that the charity faces. The team or teams which, in the opinion of JPMorgan Chase Bank, N.A. (“JPMC”), make the best attempt at developing such a solution will receive a prize for themselves and their assigned charity. All prizes are awarded “as is” and JPMC, to the fullest extent permitted by law, makes no warranty or condition in respect of any prize or part thereof. JPMC reserves the right to change any prize at its discretion.

--- Result 2 ---
SOURCE: test.txt
TYPE: txt
CONTENT:
All entries become the exclusive property of JPMC and will not be acknowledged or returned. By entering the Code for Good Hackathon (“Hackathon”), entrants accept and agree to be bound by these Hackathon Official Rules (the “Rules”) and

In [122]:
result = generate_rag_answer(
    "JPMorgan Chase Code for Good Hackathon?",
    multi_retriever,
    llm
)

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print("-", source)

ANSWER:
The JPMorgan Chase Code for Good Hackathon is an event where teams of students compete against each other on behalf of a charity assigned to them to provide a technological solution to a problem that the charity faces. The team or teams that make the best attempt at developing such a solution will receive a prize for themselves and their assigned charity.

SOURCES:
- test.txt


# negative testing

In [123]:
result = generate_rag_answer(
    "What is the capital of Australia?",
    multi_retriever,
    llm
)

print(result["answer"])

I couldn't find that information in the provided knowledge base.


# chat remembering execution

In [124]:
from app.chat.chat_service import (
    create_chat_session,
    add_message,
    format_chat_history
)

In [125]:
chat_history = create_chat_session()

print(chat_history)

[]


In [126]:
add_message(
    chat_history,
    "Who can participate in JPMC Code for Good?",
    "Students who meet the eligibility requirements can participate."
)

[{'user': 'Who can participate in JPMC Code for Good?',
  'assistant': 'Students who meet the eligibility requirements can participate.'}]

# testing the chatbot

In [129]:
from app.rag.rag_service import generate_chat_rag_answer

In [130]:
chat_history = create_chat_session()

In [131]:
result = generate_chat_rag_answer(
    "Who can participate in JPMC Code for Good?",
    multi_retriever,
    llm,
    chat_history
)

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print("-", source)

ANSWER:
The provided knowledge base context does not explicitly state who can participate in the JPMC Code for Good Hackathon. It mentions that "Teams of students will compete against each other", but it does not provide a detailed description of the eligibility criteria. 

I couldn't find that information in the provided knowledge base.

SOURCES:
- test.txt


In [132]:
add_message(
    chat_history,
    "Who can participate in JPMC Code for Good?",
    result["answer"]
)

[{'user': 'Who can participate in JPMC Code for Good?',
  'assistant': 'The provided knowledge base context does not explicitly state who can participate in the JPMC Code for Good Hackathon. It mentions that "Teams of students will compete against each other", but it does not provide a detailed description of the eligibility criteria. \n\nI couldn\'t find that information in the provided knowledge base.'}]

In [134]:
result = generate_chat_rag_answer(
    "Give about the event?",
    multi_retriever,
    llm,
    chat_history
)

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print("-", source)

ANSWER:
ABOUT THE EVENT: Teams of students will compete against each other on behalf of a charity assigned to them in order to provide a technological solution to a problem that the charity faces. The team or teams which, in the opinion of JPMorgan Chase Bank, N.A. (“JPMC”), make the best attempt at developing such a solution will receive a prize for themselves and their assigned charity.

SOURCES:
- test.txt


In [135]:
add_message(
    chat_history,
    "What are the eligibility requirements?",
    result["answer"]
)

[{'user': 'Who can participate in JPMC Code for Good?',
  'assistant': 'The provided knowledge base context does not explicitly state who can participate in the JPMC Code for Good Hackathon. It mentions that "Teams of students will compete against each other", but it does not provide a detailed description of the eligibility criteria. \n\nI couldn\'t find that information in the provided knowledge base.'},
 {'user': 'What are the eligibility requirements?',
  'assistant': 'ABOUT THE EVENT: Teams of students will compete against each other on behalf of a charity assigned to them in order to provide a technological solution to a problem that the charity faces. The team or teams which, in the opinion of JPMorgan Chase Bank, N.A. (“JPMC”), make the best attempt at developing such a solution will receive a prize for themselves and their assigned charity.'}]

In [136]:
print(format_chat_history(chat_history))

User: Who can participate in JPMC Code for Good?
Assistant: The provided knowledge base context does not explicitly state who can participate in the JPMC Code for Good Hackathon. It mentions that "Teams of students will compete against each other", but it does not provide a detailed description of the eligibility criteria. 

I couldn't find that information in the provided knowledge base.
User: What are the eligibility requirements?
Assistant: ABOUT THE EVENT: Teams of students will compete against each other on behalf of a charity assigned to them in order to provide a technological solution to a problem that the charity faces. The team or teams which, in the opinion of JPMorgan Chase Bank, N.A. (“JPMC”), make the best attempt at developing such a solution will receive a prize for themselves and their assigned charity.


# Embeddings testing

In [137]:
from app.embeddings.embedding_service import get_embeddings

In [138]:
embeddings = get_embeddings()

print(type(embeddings))

<class 'langchain_google_genai.embeddings.GoogleGenerativeAIEmbeddings'>


In [139]:
vector = embeddings.embed_query(
    "JPMC Code for Good"
)

print("Embedding dimensions:", len(vector))

Embedding dimensions: 3072


In [170]:
test_vectorstore = create_knowledge_base(
    multi_chunks[:3],
    embeddings
)

print("Vector store created successfully.")

Vector store created successfully.
